# 04 - Business Insights (KPIs & Perguntas de Negócio)

Este notebook consolida as respostas para as perguntas de negócio solicitadas no desafio técnico, utilizando a camada **Gold**.

## Perguntas que responderemos:
1. Como o negócio performou no período (Receita e Ticket Médio)?
2. Quais regiões, canais e categorias apresentam melhor desempenho?
3. Onde estão os principais gargalos operacionais (Atrasos)?
4. Qual a taxa de cancelamento e seu impacto?

In [0]:
%run ./utils

# Utils

Notebook contendo funções reutilizáveis para o pipeline de dados.

### 1. Performance Geral do Negócio (KPIs Financeiros)
Calculamos a **Receita Líquida** e o **Ticket Médio**, segmentando por mês para ver a evolução temporal.

In [0]:
spark.sql(f"""
    SELECT 
        date_format(data_pedido, 'yyyy-MM') as mes_referencia,
        COUNT(DISTINCT pedido_id) as volume_pedidos,
        ROUND(SUM(valor_liquido_item), 2) as receita_liquida,
        ROUND(SUM(valor_liquido_item) / COUNT(DISTINCT pedido_id), 2) as ticket_medio

    FROM {catalog_name}.gold.fact_vendas
    
    WHERE status_pedido != 'CANCELADO' AND flg_qualidade_venda = true
    GROUP BY 1
    ORDER BY 1
""").show()

+--------------+--------------+---------------+------------+
|mes_referencia|volume_pedidos|receita_liquida|ticket_medio|
+--------------+--------------+---------------+------------+
|       2025-01|            21|      233131.27|    11101.49|
|       2025-02|            23|      268919.77|    11692.16|
|       2025-03|            19|      303080.46|     15951.6|
|       2025-04|            30|      378962.93|     12632.1|
|       2025-05|            29|      356459.39|     12291.7|
|       2025-06|            19|      234469.04|    12340.48|
|       2025-07|            21|      250730.25|    11939.54|
|       2025-08|            26|      363731.12|    13989.66|
|       2025-09|            24|      415452.73|    17310.53|
|       2025-10|            20|      309758.53|    15487.93|
|       2025-11|            22|      295810.99|    13445.95|
|       2025-12|            20|      308617.54|    15430.88|
|       2026-01|            22|      313633.47|    14256.07|
|       2026-02|        

### 2. Desempenho por Segmentação (Região, Canal e Categoria)
Identificamos os pontos de maior sucesso e as oportunidades de melhoria.

In [0]:
print("Top 5 Regiões por Receita:")
spark.sql(f"""
    SELECT 
        v.regional_name,
        ROUND(SUM(f.valor_liquido_item), 2) as receita_liquida

    FROM {catalog_name}.gold.fact_vendas f
    
    INNER JOIN {catalog_name}.gold.dim_vendedores v ON f.vendedor_id = v.seller_id
    
    WHERE f.status_pedido != 'CANCELADO'
    GROUP BY 1 ORDER BY 2 
    DESC LIMIT 5
""").show()

print("Performance por Canal de Venda:")
spark.sql(f"""
    SELECT 
        v.nome_canal,
        ROUND(SUM(f.valor_liquido_item), 2) as receita_liquida

    FROM {catalog_name}.gold.fact_vendas f
    INNER JOIN {catalog_name}.gold.dim_vendedores v ON f.vendedor_id = v.seller_id
    
    WHERE f.status_pedido != 'CANCELADO'
    GROUP BY 1 
    ORDER BY receita_liquida DESC
""").show()

print("Top 5 Categorias de Produtos:")
spark.sql(f"""
    SELECT 
        p.category,
        ROUND(SUM(f.valor_liquido_item), 2) as receita_liquida

    FROM {catalog_name}.gold.fact_vendas f
    
    INNER JOIN {catalog_name}.gold.dim_produtos p ON f.product_id = p.product_id
    
    WHERE f.status_pedido != 'CANCELADO'
    
    GROUP BY 1 
    
    ORDER BY receita_liquida DESC 
    LIMIT 5
""").show()

Top 5 Regiões por Receita:
+-------------+---------------+
|regional_name|receita_liquida|
+-------------+---------------+
|          Sul|     1689075.22|
|      Sudeste|      837823.55|
|     Nordeste|      729265.41|
| Centro-Oeste|      729201.36|
|        Norte|      436755.08|
+-------------+---------------+

Performance por Canal de Venda:
+------------+---------------+
|  nome_canal|receita_liquida|
+------------+---------------+
|Inside Sales|     1073463.07|
|        NULL|       812786.2|
|  E-commerce|      688136.37|
| Marketplace|       624414.8|
|   Parceiros|      570168.03|
| Field Sales|      567562.84|
|    Revendas|       350551.8|
+------------+---------------+

Top 5 Categorias de Produtos:
+----------+---------------+
|  category|receita_liquida|
+----------+---------------+
|Assinatura|     1549549.48|
|  Serviços|     1112088.38|
|  Software|     1028474.17|
|  Hardware|      760445.45|
+----------+---------------+



### 3. Gargalos Operacionais (Logística e Atrasos)
Analisamos a **Taxa de Atraso** por transportadora para identificar ineficiências na entrega.

In [0]:
spark.sql(f"""
    SELECT 
        carrier_name,
        COUNT(*) as total_entregas,
        SUM(flg_atraso) as total_atrasos,
        ROUND(AVG(dias_atraso), 1) as media_dias_atraso,
        ROUND(SUM(flg_atraso) / COUNT(*) * 100, 2) as taxa_atraso_pct

    FROM {catalog_name}.gold.fact_entregas
    
    WHERE delivery_status = 'delivered'
    GROUP BY 1
    ORDER BY taxa_atraso_pct DESC
""").show()

+-------------+--------------+-------------+-----------------+---------------+
| carrier_name|total_entregas|total_atrasos|media_dias_atraso|taxa_atraso_pct|
+-------------+--------------+-------------+-----------------+---------------+
|      LogFast|            18|           14|             43.0|          77.78|
|NAO INFORMADO|             8|            4|            -13.1|           50.0|
|     TransSul|            14|            6|            -46.2|          42.86|
|    EntregaJá|             8|            2|            -48.8|           25.0|
+-------------+--------------+-------------+-----------------+---------------+



### 4. Perda de Receita (Cancelamentos)
Calculamos a taxa de cancelamento e o quanto de receita 

In [0]:
spark.sql(f"""
    SELECT 
        status_pedido,
        COUNT(DISTINCT pedido_id) as qtd_pedidos,
        ROUND(SUM(valor_liquido_item), 2) as valor_total,
        ROUND(COUNT(DISTINCT pedido_id) / FIRST(total_geral) * 100, 2) as pct_do_total

    FROM {catalog_name}.gold.fact_vendas
    
    CROSS JOIN (SELECT COUNT(DISTINCT pedido_id) as total_geral FROM {catalog_name}.gold.fact_vendas)
    GROUP BY 1
    ORDER BY 2 DESC
""").show()

+-------------+-----------+-----------+------------+
|status_pedido|qtd_pedidos|valor_total|pct_do_total|
+-------------+-----------+-----------+------------+
|     FATURADO|        100| 1403511.19|       26.18|
| EM SEPARACAO|         60|  864670.24|       15.71|
|NAO INFORMADO|         60|  886128.36|       15.71|
| EM_SEPARACAO|         59|   658531.2|       15.45|
|     ENTREGUE|         55|   553652.7|        14.4|
|    CANCELADO|         49|  634623.89|       12.83|
+-------------+-----------+-----------+------------+



## Conclusão

Com base nas consultas acima, o Analista de BI tem todos os subsídios para responder às dores do negócio.
A separação dimensional permite o cruzamento de qualquer uma das métricas acima por qualquer atributo de cliente, vendedor ou produto.